## Before You Begin

This tutorial is designed to operate in the Project 1 environment. Before you begin, do the following:
- Accept the Project 1 invite link from Learning Suite
- Configure and activate your virtual environment and project as directed in the Project 1 `README.md`
- Install the Project 1 package.
- Tell VS Code to use the correct Python interpreter: View --> Command Palette --> Python:Select Interpreter -> `Python 3.12.5 (.venv)`

The tutorial is written as a **Jupyter notebook**, which is a way of combining formatted text with small pieces of code. Run the following cell to set up the ability to do tests. You run the cell by clicking on the left-pointing triangle at the top left of the cell.


In [ ]:
# JUPYTER SETUP (run once per kernel)
import ipytest
ipytest.autoconfig()


---

### FSM definition

The textbook defines a Finite State Machine (FSM) as a tuple. Think of a _tuple_ as an ordered pair with more than two elements. The tuple for an FSA is $(S,I,s_0,f,g)$ where 
* A finite set of states $S$
* A set of input characters $I$
* A set of output characters $O$
* A start state $s_0$
* A transition function $f:S\times I \rightarrow S$
* An output function $g:S\times I \rightarrow O$


We often draw FSMs by representing each state $s\in S$ with a circle, representing the start state by a circle with an arrow pointing toward it, representing the transition function as labeled arrows connecting one state to another, and representing the output function by adding an additional label to each arrow. Arrow labels are represented as $i, o$ pairs where $i\in I$ is the input character and $o\in O$ is the output character. The label $i$ of the arrow connecting one state to another is the input that causes the FSA to _transition_ from one state to another.  The label $o$ on an arrow indicates what is _output_ while the state transition occurs.

---

### Implementations of FSMs ###

The purpose of this tutorial is to explore different ways of implementing state machines in Python.

Consider Figure 2 from Section 13.2.2 of the textbook.
I've modified Figure 2 by changing output set from $O = \{0,1\}$ to $O = \{a,b\}$ because I think it makes it easier to tell what is an input and what is an output.

<img src="./figures/Table2_Section13.2.2.png" alt="Finite state machine from Table 2 in Section13.2.2" width="400">

Let's look at three different ways to program the transition function $f: S\times I \rightarrow O$ for this FSM.

---

### Method 1: if-then structure ##

The state transition is defined as a function $f: S\times I \rightarrow S$ that maps the present state and input to a next state. If we let $s$ and $i$ denote the state of the machine and the input at the current time, respectively, and if we let $s'$ denote the next state, then we can write the next state as a function of the present state, $s' = f(s, i)$. 

We can think of this function as an if statement: 

    if the present state is s and the current input is i then the next state is s'

 We can use this idea to create a class for the FSM in the figure above. For now, we'll ignore that FSMs have output and just pay attention to the mapping from input and present state to the next state, $f: S\times I \rightarrow S$.

In [ ]:
############
## Cell 1 ##
############

class FiniteStateMachine:
    """
    Represent states with strings: 's0', 's1', 's2', 's3'
    Represent inputs with strings '0', '1'
    """
    def __init__(self) -> None:
        # Define the set of states and inputs
        # Follow the Python convention of prefixing "private" attributes with an underscore
        # Python does not enforce access restrictions, but it is useful 
        # to signal intent to others who might read your code.
        self._states: set[str] = {'s0', 's1', 's2', 's3'}
        self._inputs: set[str] = {'0', '1'}

    def get_next_state(self, present_state: 
                       str, input_symbol: str
                       ) -> str:
        """
        (present_state, input_symbol) -> next_state with no side effects
        Return the next state given the present state and input symbol.
        """
        # Validate inputs
        if present_state not in self._states:
            raise ValueError(f"Illegal present state: {present_state}")
        if input_symbol not in self._inputs:
            raise ValueError(f"Illegal input to the state machine: {input_symbol}")

                # Transitions from state s0
        if present_state == 's0':
            if input_symbol == '0':
                next_state = 's1'
            else:
                next_state = 's0'

        # Transitions from state s1
        if present_state == 's1':
            if input_symbol == '0':
                next_state = 's3'
            else:
                next_state = 's0'

        # Transitions from state s2
        if present_state == 's2':
            if input_symbol == '0':
                next_state = 's1'
            else:
                next_state = 's2'

        # Transitions from state s3
        if present_state == 's3':
            if input_symbol == '0':
                next_state = 's2'
            else:
                next_state = 's1'

        # Return next_state
        return next_state
    

The implementation is straightforward. First, it checks whether the `present_state` and `input_symbol` are valid. Then, there is an `if` block for each state. The outer `if` statement in each block checks the current state. The inner `if` statements check the input. **Run the code block** above by clicking the left-pointing triangle on the left, top side of the cell. You should see a green checkmark and the amount of time it took the cell to run. The green checkmark says that the Python interpreter running behind the scenes now has the definition of the `FiniteStateMachine` class.

Notice that the `FiniteStateMachine` class does not use any hidden state variables and has no side-effecting. As a reminder, a Python function has **side effects** if it not only generates an output that is returned but also changes one of the internal state variables. Thus, the `get_next_state()` function implements the state transition function $f: S\times I \rightarrow O$ from the mathematical definition of a FSM. The state in $S$ is specified by the `present_state` argument in the Python function `get_next_state`, the input from $I$ is specified by the `input_symbol` argument in the Python function `get_next_state`, and the output from $O$ is specified by returning `next_state`.

We want to know whether the `FiniteStateMachine` class correctly implements the FSM from the figure. That means we'll need to write tests to make sure it implements the state transition function correctly.

---

**Unit Tests** We can write unit tests for each transition. I'll show only a few examples instead of giving all possible unit tests. We have to do some setup to be able to run `pytest` inside a Jupyter notebook tutorial. 

Let's test a few of the transitions from the FSM.  Let's test transitions from state `'s0'` and state `'s1'`. The figure represents the transition function. For state `'s0'` we need to check the following:
* $f(\text{`s0'}, 0) = \text{`s1'}$ is represented by the arrow from $s_0$ to $s_1$.
* $f(\text{`s0'}, 1) = \text{`s0'}$ is represented by the arrow from $s_0$ back to $s_0$.

Similarly, we'll test that the transitions from state `'s1'` in the function `get_next_state` satisfy:
* $f(\text{`s1'}, 0) = \text{`s3'}$ is represented by the arrow from $s_1$ to $s_3$.
* $f(\text{`s1'}, 1) = \text{`s0'}$ is represented by the arrow from $s_1$ to $s_0$.

We'll use `pytest` and the pattern shown in Homework 2 for testing classes. The pattern is

1. **Instantiate the object** – Create a fresh instance of the class so that you have full control over its state.
2. **Specify the inputs (domain elements)** – Set both:
   - the explicit method argument(s), and
   - any hidden inputs stored in member variables.
3. **Specify the expected outputs (codomain elements)** – Write down what you expect the method to return *and* what you expect the internal state to become after the method runs.
4. **Call the method** – Execute the method with the specified input.
5. **Assert on the explicit output** – Compare the return value with your expected result.
6. **Assert on the implicit output** – Compare the modified internal state (object fields) with your expected result.

Note that if I were doing this for an assignment, I'd write the first few tests based on the math, work out the math for other tests, and then I'd use an AI tool to write the tests for the math I've done.

In [ ]:
%%ipytest -qq
# ----------------------------
# Transition tests for FSM
# ----------------------------


def test_s0_on_0_goes_to_s1() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = FiniteStateMachine()

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "0"
    current_state: str = "s0"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s1"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s0_on_1_goes_to_s0() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = FiniteStateMachine()

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "1"
    current_state: str = "s0"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s0"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s1_on_0_goes_to_s3() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = FiniteStateMachine()

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "0"
    current_state: str = "s1"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s3"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s1_on_1_goes_to_s0() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = FiniteStateMachine()

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "1"
    current_state: str = "s1"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s0"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state

You run the tests by clicking the run button in the Jupyter notebook tutorial. If you see green dots then the tests passed. Since there are four tests, you should see four green dots.

Writing every possible test is tedious, and it's a waste of effort to cut and paste so much code that is nearly identical. In Project 3, we'll talk about **parameterized tests**, which makes it more economical to write a bunch of tests.

---

**Unit Tests From Traces.** Let's write a different kind of unit test. Suppose the input to the FSM is `01000`. We "do the math" by tracing the transitions in the FSM by hand. The state sequence produced by the input above should be s0 -> s1 -> s0 -> s1 -> s3 -> s2. 
In this case, "the math" is knowing how an FSM works, choosing a sequence of input characters, and looking at what states are visited as the FSM reads through each input character. 

Let's look at how the FSM runs. The following cell is more about understanding the math then using it to write a test. It focuses on showing how a character from the `input` is read for each `present_state` to produce a `next_state`. We then set the `present_state` in the FSM to the `next_state` and run another round.

In [ ]:
############
## Cell 2 ##
############

fsm: FiniteStateMachine = FiniteStateMachine()

# Explicitly track state outside the FSM (since the transition function has no side effects)
current_state: str = 's0'

input_sequence: list[str] = ['0', '1', '0', '0', '0']

print(f"Start state is {current_state}")
for symbol in input_sequence:
    next_state = fsm.get_next_state(current_state, symbol)
    print(f"Present state {current_state} and input {symbol} -> next state {next_state}")
    current_state = next_state

print(f"Final state is {current_state}")

We can put this into a unit test that will run in `pytest`. I used AI to write this test since (a) I had already "done the math" and (b) I had run the code to make sure I understood how things worked. Here's the unit test that demonstrates the idea of a **trace**. In this context, the word "trace" means the sequence of states visited.

Run the cell to make sure that the test passes.

In [ ]:
%%ipytest -qq
def test_unit_state_sequence_for_01000() -> None:
    """
    Integration test:
    Input sequence '0 1 0 0 0' should traverse the states
    s0 -> s1 -> s0 -> s1 -> s3 -> s2
    """

    # ---------------------------
    # Step 1: Instantiate the FSM
    # ---------------------------
    fsm = FiniteStateMachine()

    ################################
    ## Specify inputs from domain ##
    ################################
    # Step 2: Specify the input sequence (as list of symbols)
    input_sequence: list[str] = ['0', '1', '0', '0', '0']

    ##############################################
    ## Specify expected outputs in the codomain ##
    ##############################################
    # Step 3: Specify the expected state *trace* including start and each next state
    # Start in s0, then after each symbol we expect:
    # s0 --0--> s1 --1--> s0 --0--> s1 --0--> s3 --0--> s2
    expected_trace: list[str] = ['s0', 's1', 's0', 's1', 's3', 's2']
    expected_final_state: str = 's2'

    # ------------------------------------
    # Step 4: Execute the FSM step-by-step
    # ------------------------------------
    current_state: str = 's0'
    observed_trace: list[str] = [current_state]  # include start state
    for symbol in input_sequence:
        next_state: str = fsm.get_next_state(current_state, symbol)  # function under test
        observed_trace.append(next_state)  # record the next state
        current_state = next_state         # advance the machine

        # Optional sanity check per step (keeps the "math" visible while we run)
        assert current_state in {'s0', 's1', 's2', 's3'}

    #######################################
    ## Check values returned by function ##
    #######################################
    # Step 5: Check the full state trace
    assert observed_trace == expected_trace

    # Step 6: Check final state explicitly
    assert current_state == expected_final_state

Since there was only one test, the notebook shows a single green green dot.

Recall that the transition function $f:S\times I \rightarrow S$ is a **function** (not a _partial function_). That means every input must be mapped to an output. The inputs are state-action pairs $(s, i)$. We can write a unit test that checks whether the FSM defined in the code has a transition function that is actually a function.

In [ ]:
%%ipytest -qq
# ----------------------------
# FSM transition is a function 
# ----------------------------

def test_transitions_defined_for_every_state_and_input() -> None:
    # Step 1: Instantiate
    fsm = FiniteStateMachine()

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    states: set[str] = {"s0", "s1", "s2", "s3"}
    inputs: set[str] = {"0", "1"}

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    # Every transition should land in a valid state.
    valid_states: set[str] = states.copy()

    # ------------------------------------
    # Step 4: Execute and collect results #
    # ------------------------------------
    for present_state in states:
        for input_symbol in inputs:
            # Must not raise, and must return a valid next state
            next_state = fsm.get_next_state(present_state, input_symbol)

            ###############################################
            ## Step 5: Check values returned by function ##
            ###############################################
            assert next_state in valid_states, (
                f"Undefined transition: ({present_state!r}, {input_symbol!r}) "
                f"returned invalid next state {next_state!r}"
            )

            # Optional: determinism check — same input from same state yields same next state
            assert fsm.get_next_state(present_state, input_symbol) == next_state, (
                f"Nondeterministic transition from ({present_state!r}, {input_symbol!r})"
            )

---

**What's Wrong with this Code?** Although conceptually simple, the code is not modular and would be very difficult to maintain if the state machine were very large. As an example, a colleague worked for a company that implemented a FSM as part of a website. The FSM had hundreds of transitions. Nobody dared changed the code for fear of messing up the code.

Let's try an approach that is also conceptually simple because it uses the transition table from the book. The approach is a bit more general because it allows us to instantiate any FSM of our choosing.

---
---

### Method 2: State transition table ###

Let's represent this FSM using something like Table 2 from the textbook. We'll split the table into two pieces: a table that handles state transitions and a table that handles outputs. The first table is a present-state, next-state table. It implements the finite state machine mapping $f:S\times I \rightarrow S$. 

I typed "how do i represent a present state next state table for a finite state machine in python?" into copilot and am using a version of that code that I modifed in a bunch of ways (too boring to list).

The basic building block of this implementation is the _transition_ method, which is implemented as a dictionary. More precisely, it is a dictionary of dictionaries. More on that later.

In [ ]:
############
## Cell 3 ##
############

class FiniteStateMachine:
    """
    A table-driven, side-effect-free FSM.

    - No internal 'present state' is stored.
    - Transitions are queried via get_next_state(present_state, input_symbol).
    - Valid states and inputs are provided at construction.
    """

    def __init__(self, states: set[str], inputs: set[str]) -> None:
        self._states: set[str] = set(states)
        self._inputs: set[str] = set(inputs)
        # transition table: present_state -> (input_symbol -> next_state)
        self._transitions: dict[str, dict[str, str]] = {}

    def add_transition(self, state: str, input_symbol: str, next_state: str) -> None:
        """Register a transition (state, input_symbol) -> next_state with validation."""
        if state not in self._states:
            raise ValueError(f"Unknown state in transition: {state!r}")
        if input_symbol not in self._inputs:
            raise ValueError(f"Unknown input in transition: {input_symbol!r}")
        if next_state not in self._states:
            raise ValueError(f"Unknown next state in transition: {next_state!r}")

        if state not in self._transitions:
            self._transitions[state] = {}
        self._transitions[state][input_symbol] = next_state

    def get_next_state(self, present_state: str, input_symbol: str) -> str:
        """
        Transition function: (present_state, input_symbol) -> next_state.
        Performs validation and does not mutate internal state.
        """
        if present_state not in self._states:
            raise ValueError(f"Illegal present state: {present_state!r}")
        if input_symbol not in self._inputs:
            raise ValueError(f"Illegal input symbol: {input_symbol!r}")

        try:
            return self._transitions[present_state][input_symbol]
        except KeyError:
            raise KeyError(
                f"No transition defined for state {present_state!r} on input {input_symbol!r}"
            )

    def show_table(self) -> None:
        """Pretty-print the transition table."""
        header = f"{'Present State':^15} | {'Input':^8} | {'Next State':^12}"
        line = "_" * len(header)
        print(header)
        print(line)
        for present_state, row in self._transitions.items():
            for input_symbol, next_state in row.items():
                print(f"{present_state:^15} | {input_symbol:^8} | {next_state:^12}")

Let's discuss the line 

    if self._present_state in self._transitions 

Suppose that you have a Python dictionary D = {'a': 1, 'b': 2}. The keys in the dictionary are 'a' and 'b'. When you write 

    if 'a' in D: 

you are asking whether 'a' is in the keys of the dictionary. Let's confirm with an aside.

In [ ]:
D: dict[str, int] = {'a':1, 'b': 2}
if 'a' in D: 
    print('\'a\' is in D')
else:
    print('\'a\' is not in D')

Try with something that is not a key in the dictionary.

In [ ]:
D: dict[str, int] = {'a':1, 'b': 2}
if 'c' in D: 
    print('\'c\' is in D')
else:
    print('\'c\' is not in D')

Returning to the FSM, let's instantiate the class and add the state transitions for the FSM shown above.

In [ ]:
############
## Cell 4 ##
############

# Example usage
fsm: FiniteStateMachine = FiniteStateMachine(states={'s0', 's1', 's2', 's3'}, inputs={'0', '1'})

fsm.add_transition('s0', '0', 's1')
fsm.add_transition('s0', '1', 's0')
fsm.add_transition('s1', '0', 's3')
fsm.add_transition('s1', '1', 's0')
fsm.add_transition('s2', '0', 's1')
fsm.add_transition('s2', '1', 's2')
fsm.add_transition('s3', '0', 's2')
fsm.add_transition('s3', '1', 's1')


In [ ]:
print(fsm._transitions)

We instantiated the class in the first line, and at the same time we told it the start state. 

Let's talk about what each call to _add\_transition_ does. Recall from the definition of a FSM (section 13.2.2, def 1 of the textbook) that a transition is a function that maps present states and inputs to next states, $f: S\times I \rightarrow S$. When the domain and co-domain are finite, we can represent functions as a set of tuples. The tuple $(s_0, 0, s_1)$ represents what happens to the function $f$ when we pass it $s0$ and $0$. In other words, $f(s_0,0) = s_1$. Stated simply, the first two elements of each tuple are the present state and input, respectively, and the output is the next state. Thus, each tuple is (present state, input, next state).

Following the textbook, we can represent the transition function using a present state, next state table. What does the present-state, next-state table look like?

In [ ]:
############
## Cell 5 ##
############

fsm.show_table()

Although in a different format than Table 2 in Section 13.2.2, it contains the same information.

---

**Unit Tests.** We can modify the unit tests we wrote above with one change: we have to tell the constructor what the set of states and inputs are.

In [ ]:
%%ipytest -qq
# ----------------------------
# Transition tests for FSM
# ----------------------------

## Each test must instantiate the FSM with states and inputs
## Let's use a global FSM object for this purpose
fsm: FiniteStateMachine = FiniteStateMachine(states={'s0', 's1', 's2', 's3'}, inputs={'0', '1'})

fsm.add_transition('s0', '0', 's1')
fsm.add_transition('s0', '1', 's0')
fsm.add_transition('s1', '0', 's3')
fsm.add_transition('s1', '1', 's0')
fsm.add_transition('s2', '0', 's1')
fsm.add_transition('s2', '1', 's2')
fsm.add_transition('s3', '0', 's2')
fsm.add_transition('s3', '1', 's1')

def test_s0_on_0_goes_to_s1_v2() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = fsm # Use global FSM object

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "0"
    current_state: str = "s0"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s1"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s0_on_1_goes_to_s0_v2() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = fsm # Use global FSM object

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "1"
    current_state: str = "s0"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s0"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s1_on_0_goes_to_s3_v2() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = fsm # Use global FSM object

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "0"
    current_state: str = "s1"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s3"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state


def test_s1_on_1_goes_to_s0_v2() -> None:
    # Step 1: Instantiate
    m: FiniteStateMachine = fsm # Use global FSM object

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    input_symbol: str = "1"
    current_state: str = "s1"

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    expected_next_state: str = "s0"

    # Step 7: Call object method
    returned = m.get_next_state(current_state, input_symbol)

    ###############################################
    ## Step 4: Check values returned by function ##
    ###############################################
    assert returned == expected_next_state

Four green dots should appear when you run the cell.

There is a hole in the unit tests that we have written. The transition function $f:S\times I\rightarrow O$ must be a **function**. It can't be a **partial function** and it can't have an input map to two places (which would make it "not a function"). The second condition is impossible since we used a dictionary to represent the state transitions and Python dictionaries don't allow repeated keys. We really should check to see if the state transition table is a function or just a partial function. 

In [ ]:
%%ipytest -qq
# ----------------------------
# Function vs Partial Function
# ----------------------------

def test_fsm_is_total_function_over_states_cross_inputs() -> None:
    """
    Verify that the FSM defines a *total function* S x Σ -> S:
    For every (state, input) pair, get_next_state returns a next state
    (i.e., no KeyError is raised).
    """

    # Step 1: Instantiate FSM and add transitions (the full table)
    states = {'s0', 's1', 's2', 's3'}
    inputs = {'0', '1'}
    m = FiniteStateMachine(states=states, inputs=inputs)

    m.add_transition('s0', '0', 's1')
    m.add_transition('s0', '1', 's0')
    m.add_transition('s1', '0', 's3')
    m.add_transition('s1', '1', 's0')
    m.add_transition('s2', '0', 's1')
    m.add_transition('s2', '1', 's2')
    m.add_transition('s3', '0', 's2')
    m.add_transition('s3', '1', 's1')

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    # Domain is S x Σ; we'll iterate all pairs
    domain_pairs = [(s, a) for s in states for a in inputs]

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    # Codomain is S; we just need "next_state ∈ S" and no missing mappings

    # Step 4: Exercise and assert totality
    observed_pairs = set()
    for s, a in domain_pairs:
        next_state = m.get_next_state(s, a)  # should not raise
        assert next_state in states
        observed_pairs.add((s, a))

    # Step 5: Every S x Σ pair was successfully mapped
    assert len(observed_pairs) == len(states) * len(inputs)


def test_fsm_is_only_partial_function_when_a_transition_is_missing() -> None:
    """
    Create an FSM missing at least one transition; show it’s *partial*:
    Some (state, input) pairs map to a next state, but at least one pair is undefined.
    """

    # Step 1: Instantiate FSM and add an *incomplete* transition table
    states = {'s0', 's1', 's2', 's3'}
    inputs = {'0', '1'}
    m = FiniteStateMachine(states=states, inputs=inputs)

    m.add_transition('s0', '0', 's1')
    m.add_transition('s0', '1', 's0')
    m.add_transition('s1', '0', 's3')
    m.add_transition('s1', '1', 's0')
    m.add_transition('s2', '0', 's1')
    m.add_transition('s2', '1', 's2')
    m.add_transition('s3', '0', 's2')
    # NOTE: Intentionally omit ('s3','1') -> 's1' to make it partial

    #######################################
    ## Step 2: Specify inputs from domain #
    #######################################
    domain_pairs = [(s, a) for s in states for a in inputs]

    #################################################
    ## Step 3: Specify expected outputs in codomain #
    #################################################
    # Expect: at least one missing mapping (raises KeyError)

    # Step 4: Exercise and classify
    defined_count = 0
    missing_pairs: list[tuple[str, str]] = []

    for s, a in domain_pairs:
        try:
            next_state = m.get_next_state(s, a)
            assert next_state in states
            defined_count += 1
        except KeyError:
            missing_pairs.append((s, a))

    # Step 5: Assert it is *partial*: some pairs defined, at least one missing
    assert defined_count > 0, "No pairs were defined—this would not be a useful FSM."
    assert len(missing_pairs) >= 1, "Expected at least one missing (state, input) mapping."
    # (Optional) sanity check: the missing one should be ('s3','1') per our setup
    assert ('s3', '1') in missing_pairs

**Trace-based Unit Test.**
If the input is 01001 the state sequence should be s0 -> s1 -> s0 -> s1 -> s2. Let's first just do a print out to make sure we see what we expect.

In [ ]:
############
## Cell 6 ##
############

input_sequence: list[str] = ['0', '1', '0', '0', '0']

# Start explicitly in s0
current_state: str = 's0'

print(f"The start state is {current_state}")
for symbol in input_sequence:
    next_state: str = fsm.get_next_state(current_state, symbol)
    print(f"Present state: {current_state}, Input: {symbol}, Next state: {next_state}")
    # Advance state externally
    current_state = next_state

print(f"The final state is {current_state}")

We can write an actual integration test using the math that we just did.

In [ ]:
%%ipytest -qq
def test_state_sequence_for_01000_v2() -> None:
    """
    Unit test:
    Input sequence '0 1 0 0 0' should traverse the states
    s0 -> s1 -> s0 -> s1 -> s3 -> s2
    """

    # ---------------------------
    # Step 1: Instantiate the FSM
    # ---------------------------
    fsm = FiniteStateMachine(states={'s0', 's1', 's2', 's3'}, inputs={'0', '1'})

    # Register the transition table
    fsm.add_transition('s0', '0', 's1')
    fsm.add_transition('s0', '1', 's0')
    fsm.add_transition('s1', '0', 's3')
    fsm.add_transition('s1', '1', 's0')
    fsm.add_transition('s2', '0', 's1')
    fsm.add_transition('s2', '1', 's2')
    fsm.add_transition('s3', '0', 's2')
    fsm.add_transition('s3', '1', 's1')

    ################################
    ## Specify inputs from domain ##
    ################################
    # Step 2: Specify the input sequence (as list of symbols)
    input_sequence: list[str] = ['0', '1', '0', '0', '0']

    ##############################################
    ## Specify expected outputs in the codomain ##
    ##############################################
    # Step 3: Specify the expected state *trace* including start and each next state
    # Start in s0, then after each symbol we expect:
    # s0 --0--> s1 --1--> s0 --0--> s1 --0--> s3 --0--> s2
    expected_trace: list[str] = ['s0', 's1', 's0', 's1', 's3', 's2']
    expected_final_state: str = 's2'

    # ------------------------------------
    # Step 4: Execute the FSM step-by-step
    # ------------------------------------
    current_state: str = 's0'
    observed_trace: list[str] = [current_state]  # include start state
    for symbol in input_sequence:
        next_state: str = fsm.get_next_state(current_state, symbol)  # function under test
        observed_trace.append(next_state)  # record the next state
        current_state = next_state         # advance the machine

        # Optional sanity check per step (keeps the "math" visible while we run)
        assert current_state in {'s0', 's1', 's2', 's3'}

    #######################################
    ## Check values returned by function ##
    #######################################
    # Step 5: Check the full state trace
    assert observed_trace == expected_trace

    # Step 6: Check final state explicitly
    assert current_state == expected_final_state

A single green dot. Whew!

The key idea in this implementation is to represent the state transition as a dictionary of dictionaries. The out dictionary is keyed by the present state. Then, the inner dictionary is keyed by the input. 

This isn't a bad implementation, but when we use FSMs in Project 1, we'll want to be able to write really modular code. Let's explore a new approach.

---

### Method 3: States as Functions ###

The transition function maps state-input pairs to next states. The `get_next_state` method takes a state and an input and produces the next state. What if we "let the state speak for itself", meaning we act like the circles in an FSM figure can decide for themselves where they should transition given specific inputs. We can treat these circles like little functions themselves.

The key idea in this third method for representing FSMs is to represent each state as a function and then ask it what should happen when we pass it a given input. For example, we can represent state $s_0$ as a function and then ask "state $s_0$, what should the next state be if the input is $i$?" Before giving example code, we need to review a programing concept.

This approach changes our definition of a state machine. The new definition (ignoring the output) is:
* A set of input characters $I$
* A set of output characters $O$
* A set of states $S$ where each state $s\in S$ is a function that maps an input to the next state and an output
  - $s: I \rightarrow S\times O$.
* A start state $s_0$

Notice that there is no transition function. Instead, we let each state $s$ encode its own transition function. We'll have to do some thinking about what this means and what it would look like in Python. We'll start with a review.

---




Recall from CS 111 that **everything is an object** in Python: strings, integers, classes, and functions. And every object is assigned to a location in memory. Let me illustrate with a simple function.

In [ ]:
def square(x: float) -> float: return x*x

Let's inspect this function.

In [ ]:
print(square)

When you run the cell above you should see something like

    <function square at 0x1158cc2c0>

Let's break down what this means.
- The first thing you see inside the angle brackets is the word _function_. This simply says that the type of the object is a function.
- The second thing you see is the word _square_. This simply says that the name of the function is "square".
- The next thing you see is _at 0x..._. The _0x_ tells you that you're going to see a number written in hexadecimal. (Think of the _x_ in _0x_ as the third letter of hexadecimal.)  The numbers and letters after the _0x_ are the address in memory in which the function is stored. Your numbers will be different than what I show above because the address is assigned when we create the function and depends on what else is in memory on your computer.

Python lets us pass around objects including _function objects_. That means we can pass a function object to another function, return a function from another function, etc. Let's illustrate by defining functions that compute the square, cube, and fourth power of an input. We'll then create a function that takes a string as an input and returns one of the functions. That's really abstract, so take a look at the code and then we'll discuss some more.

In [ ]:
# This import gives us the type hint "function". It is renamed from "Callable"
from typing import Callable as function

def square(x: float) -> float: return x*x
def cube(x: float) -> float: return x*x*x
def quad(x: float) -> float: return x*x*x*x

def pick_a_function(choice: str) -> function[[float], float]:
    if choice == "square": 
        return square
    elif choice == "cube": 
        return cube
    elif choice == "quad": 
        return quad
    else: 
        raise ValueError(choice = " is not a valid function type")


Note the import at the top of the block. It just defines the keyword _function_ so that we can tell Python that _pick\_a\_function_ returns a Python function. Let's break down what the following line of code means.

    def pick_a_function(choice: str) -> function[[float], float]:

The line says that we are defining a function called _pick\_a\_function_ that takes a string named _choice_ as input and returns a function. The things in the square brackets next to function

    function[[float], float]

tells us type information about the function being returned. More specifically, the function that is being returned takes as input a float, which is indicated by the _[float]_ and returns an output that is also a float, which is indicated by the _float_ not in brackets. 



Let's call _pick\_a\_function_, observe that it returns a function, and then call the function it calls.

In [ ]:
my_function = pick_a_function("square")
print(f"The function object returned is: {my_function}")
print(f"When I run the function on an input of 2 it returns: {my_function(2)}")


The _square_ function was returned. Let's try the other options as well.

In [ ]:
my_function = pick_a_function("cube")
print(my_function)
print(f"my function returned {my_function(2)}")

my_function = pick_a_function("quad")
print(my_function)
print(f"my function returned {my_function(2)}")

So, functions can call and return functions. That can be confusing, but we can use it to define a different kind of FSM. The key idea is that we'll treat each state as a function that takes an input and returns the next state. Let's use that idea to write a FSM that treats states like functions.



Referring to the figure of the FSM above, our job is to create a function _s0_ that we can ask "what will you do on a specific input?" In other words, we let $s_0$ make its own decisions. We pass it an input and it decides where it will go. For now, let's only ask it about the **state transitions**. We'll add the outputs in a later cell.

In [ ]:
############
## Cell 7 ##
############

# This import allows us to return a "function". 
# It is renamed from "Callable"
from typing import Callable as function

class FiniteStateMachine:
    def __init__(self) -> None:
        self._states: set[function[[str], function]] = {self.s0, self.s1, self.s2, self.s3}
        self._inputs: set[str] = {'0', '1'}
    
    ######################################
    ## Define a Function for each state ##
    ######################################
    def s0(self, input_symbol: str) -> function[[str],function]:
        next_state: function
        if input_symbol == "0": 
            next_state = self.s1
        elif input_symbol == "1": 
            next_state = self.s0
        else: 
            raise ValueError("Illegal input to the state machine " + input_symbol)
        return next_state
    
    def s1(self, input_symbol: str) -> function[[str],function]:
        next_state: function
        if input_symbol == "0": 
            next_state = self.s3
        elif input_symbol == "1": 
            next_state = self.s0
        else: 
            raise ValueError("Illegal input to the state machine " + input_symbol)
        return next_state
    
    def s2(self, input_symbol: str) -> function[[str],function]:
        next_state: function
        if input_symbol == "0": 
            next_state = self.s1
        elif input_symbol == "1": 
            next_state = self.s2
        else: 
            raise ValueError("Illegal input to the state machine " + input_symbol)
        return next_state
    
    def s3(self, input_symbol: str) -> function[[str],function]:
        next_state: function
        if input_symbol == "0": 
            next_state = self.s2
        elif input_symbol == "1": 
            next_state = self.s1
        else: 
            raise ValueError("Illegal input to the state machine " + input_symbol)
        return next_state

Notice that this code is almost identical to the if-then-else structure. The only differences are
 - states are represented as functions and not as strings
 - the code is modular

 Let's see if it works. Consider going through the state transitions by hand for the input 01000. The state sequence should be s0 -> s1 -> s0 -> s1 -> s3 -> s2.

In [ ]:
############
## Cell 8 ##
############

fsm: FiniteStateMachine = FiniteStateMachine()

input_sequence: list[str] = ['0', '1', '0', '0', '0']

present_state: function[[str], function]
next_state: function[[str], function]

# Start explicitly in s0
present_state = fsm.s0

print(f"The start state is {present_state}")
for symbol in input_sequence:
    next_state = present_state(symbol) # Ask the present state what the next state should be
    print(f"Present state: {present_state}, Input: {symbol}, Next state: {next_state}")
    # Advance state externally
    present_state = next_state

The state sequence is correct. Of course, the things that are printed out are not strings but rather functions along with information about to whom those functions belong. The word "function" has been replaced by "bound method", which is just Python's way of saying that functions that exist inside classes are called "methods" and they are _bound_ (i.e., belong to) an instantiated class.

We can use a special property in Python objects, which is that every Python object has to have a name. The name is called `object.__name__`. Let's print out the names instead of the clunky object references just to make sure things are what we expect. 


In [ ]:
############
## Cell 9 ##
############

fsm: FiniteStateMachine = FiniteStateMachine()

input_sequence: list[str] = ['0', '1', '0', '0', '0']

present_state: function[[str], function]
next_state: function[[str], function]

# Start explicitly in s0
present_state = fsm.s0

print(f"The start state is {present_state.__name__}")
for symbol in input_sequence:
    next_state = present_state(symbol) # Ask the present state what the next state should be
    print(f"Present state: {present_state.__name__}, Input: {symbol}, Next state: {next_state.__name__}")
    # Advance state externally
    present_state = next_state

I'd use this property to write unit and integration tests. But rather than doing that, let's talk about outputs.

---

### Outputs ###

In the previous version, we let each state *speak for itself*: each state was represented as a function, and when we passed in an input symbol, it told us what the next state should be. That is, every state acted like its own little transition function.  

But FSMs don’t just have to know where to go next — they also need to produce **outputs**. The modification we make is simple: instead of just returning the next state, a state function also returns an output.  

Formally, instead of defining each state as a function that only tells us the next state, $s:I \rightarrow S$, we now define each state as a function that tells us both the next state and the output. This means that each state is $s: I \rightarrow S\times O$. The Cartesian product on the righthand side of the arrow tells us that the function produces a (`next_state`, `output`) tuple. 

This means:  
- Each state still takes an input symbol from the input set `I`.  
- But now, for each input, the state produces **two things**:  
  1. the **next state** in `S`, and  
  2. the **output** in `O`.  


**Why This Matters**

This connects to the core definition of finite state machines: they are not only about **transitions** but also about **outputs**. The “speaking state” model makes this explicit. A state, when asked about an input, has two responsibilities:  

1. Tell us where the machine should go next (the next state).  
2. Tell us what output should be produced at the same time.  

This reinforces the idea that FSMs are fundamentally **two-layered**:  
- **State transitions**: how the machine moves from one configuration to another.  
- **Outputs**: the visible behavior we get from the machine as it processes inputs.  

**Defining a Type for States that are Functions**

Let's define a State data type so we dont' have to write things like `function[[str], tuple[function, str]]` for a return type. A state is a function that maps input characters to a tuple of (`next_state`, `output`), so we'll use 

  `State= function[[str], tuple["State", str]]`

In [ ]:
############
## Cell 8 ##
############

# A state is a function: input_symbol -> (next_state_fn, output)
State= function[[str], tuple["State", str]]

class FiniteStateMachine:
    def __init__(self) -> None:
        self._states: set[State] = {self.s0, self.s1, self.s2, self.s3}
    
    ######################################
    ## Define a Function for each state ##
    ######################################
    def s0(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state = self.s1
            output = "b"
        elif input_symbol == "1":
            next_state = self.s0
            output = "a"
        else:
            raise ValueError(f"Illegal input: {input_symbol!r}")
        return next_state, output

    def s1(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state = self.s3
            output = "b"
        elif input_symbol == "1":
            next_state = self.s0
            output = "b"
        else:
            raise ValueError(f"Illegal input: {input_symbol!r}")
        return next_state, output

    def s2(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state = self.s1
            output = "a"
        elif input_symbol == "1":
            next_state = self.s2
            output = "b"
        else:
            raise ValueError(f"Illegal input: {input_symbol!r}")
        return next_state, output

    def s3(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state = self.s2
            output = "a"
        elif input_symbol == "1":
            next_state = self.s1
            output = "a"
        else:
            raise ValueError(f"Illegal input: {input_symbol!r}")
        return next_state, output

We'll just demonstrate how this works, rather than writing tests.

In [ ]:
fsm: FiniteStateMachine = FiniteStateMachine()

input_sequence: list[str] = ['0', '1', '0', '0', '0']
output_sequence: list[str] = []

# Start in s0 (a function)
present_state: State= fsm.s0

print(f"Start state is {present_state.__name__}")
for symbol in input_sequence:
    next_state, output = present_state(symbol)  # ask current state for (next_state, output)
    print(
        f"Present state: {present_state.__name__}, "
        f"Input: {symbol}, "
        f"Next state: {next_state.__name__}, "
        f"Output: {output}"
    )
    output_sequence.append(output)
    present_state = next_state  # advance

print(output_sequence)

---

Let's recap what we've done.
- We implemented a FSM using if-then-else statements. This was based on the mental model that says the transition function operates like "if the present state is _s_ and the input is _i_ then the next state should be _s'_ ".
- We implemented a FSM by constructing a state transition table. This was based on the mental model that says each transition can be encoded as a tuple _(present state, input, next state)_.
- We implemented a FSM by creating a function for each state. This was based on the mental model that says "we can implement the transition function by asking each state (i.e., calling each state function) what its next state and output should be for a given input".

We demonstrated that each of these approaches works fine. I like the last one because (a) it is modular and (b) it allows me to take a state diagram that I've drawn and turn it directly into code. 

When we apply FSMs to project 1, we'll want a generic method that manages our FSM. Specifically, we'll create a method _run_ that
- _manages_ the input 
- _tracks_ the present state
- _runs_ the FSM by asking each state function about the next state and output for a given input
- _sets_ the present state to the next state
- _returns_ the output


In [ ]:
############
## Cell 9 ##
############

from typing import Callable as function

VERBOSE = True

# A state is a function that takes an input symbol and returns (next_state_function, output)
State = function[[str], tuple["State", str]]

class StateMachine:
    def __init__(self) -> None:
        self.initial_state: State = self.s0

    ######################################
    ## Define a Function for each state ##
    ######################################
    def s0(self, input_symbol: str) -> tuple[State, str]:
        # return (next_state_function, output_char)
        if input_symbol == "0":
            next_state: State = self.s1
            output: str = "b"
        elif input_symbol == "1":
            next_state = self.s0
            output = "a"
        else:
            raise ValueError(f"Illegal input to the state machine {input_symbol!r}")
        return next_state, output

    def s1(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state: State = self.s3
            output: str = "b"
        elif input_symbol == "1":
            next_state = self.s0
            output = "b"
        else:
            raise ValueError(f"Illegal input to the state machine {input_symbol!r}")
        return next_state, output

    def s2(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state: State = self.s1
            output: str = "a"
        elif input_symbol == "1":
            next_state = self.s2
            output = "b"
        else:
            raise ValueError(f"Illegal input to the state machine {input_symbol!r}")
        return next_state, output

    def s3(self, input_symbol: str) -> tuple[State, str]:
        if input_symbol == "0":
            next_state: State = self.s2
            output: str = "a"
        elif input_symbol == "1":
            next_state = self.s1
            output = "a"
        else:
            raise ValueError(f"Illegal input to the state machine {input_symbol!r}")
        return next_state, output


#############################################
## Define a function that runs the machine ##
#############################################
def run(input_sequence: list[str], fsm: StateMachine) -> list[str]:
    present_state: State = fsm.initial_state
    output_sequence: list[str] = []

    for symbol in input_sequence:
        before_state = present_state                 # capture for logging
        next_state, output = present_state(symbol)   # ask current state for (next, output)
        output_sequence.append(output)               # record output
        present_state = next_state                   # advance

        if VERBOSE:
            print(
                f"Present state: {before_state.__name__}, "
                f"Input: {symbol}, "
                f"Next state: {next_state.__name__}, "
                f"Output: {output}"
            )

    return output_sequence

The code above tries to use good modular design. The _StateMachine_ class defines an initial state and all transitions. The _run_ method just does what we've been doing in class: it steps through each state consuming inputs and collecting outputs.

Let's run the fsm on the same input as above.

In [ ]:
fsm: StateMachine = StateMachine()
print(run(['0', '1', '0', '0', '0'], fsm))


---

### Testing ###

We can use assert statements to test whether the FSM acts like we expect. We'll collect all the assert statements into a single test function. You can ask a LLM what the following code is doing if you've never used assert statements. You can also ask a LLM to explain the difference between a unit test and an integration test.

I created the tests by stepping through the FSM by hand.

In [ ]:
%%ipytest -qq

def test_fsm():
    ################
    ## Unit tests ##
    ################

    # Transitions from state s0 with input '0'
    next_state, output = fsm.s0('0')
    assert output == 'b', "Output for f(s0, 0) failed"
    assert next_state == fsm.s1, "Next state for f(s0,0) failed"

    # Transitions from state s0 with input '1'
    next_state, output = fsm.s0('1')
    assert output == 'a', "Output for f(s0, 1) failed"
    assert next_state == fsm.s0, "Next state for f(s0,1) failed"

    # Transitions from state s1 with input '0'
    next_state, output = fsm.s1('0')
    assert output == 'b', "Output for f(s1, 0) failed"
    assert next_state == fsm.s3, "Next state for f(s1,0) failed"

    # Transitions from state s1 with input '1'
    next_state, output = fsm.s1('1')
    assert output == 'b', "Output for f(s1, 1) failed"
    assert next_state == fsm.s0, "Next state for f(s1,0) failed"

    # Transitions from state s2 with input '0'
    next_state, output = fsm.s2('0')
    assert output == 'a', "Output for f(s2, 0) failed"
    assert next_state == fsm.s1, "Next state for f(s2,0) failed"

    # Transitions from state s2 with input '1'
    next_state, output = fsm.s2('1')
    assert output == 'b', "Output for f(s2, 1) failed"
    assert next_state == fsm.s2, "Next state for f(s2,0) failed"

    # Transitions from state s3 with input '0'
    next_state, output = fsm.s3('0')
    assert output == 'a', "Output for f(s3, 0) failed"
    assert next_state == fsm.s2, "Next state for f(s3,0) failed"

    # Transitions from state s3 with input '1'
    next_state, output = fsm.s3('1')
    assert output == 'a', "Output for f(s3, 1) failed"
    assert next_state == fsm.s1, "Next state for f(s3,0) failed"

    ################
    ## Unit tests ##
    ################
    assert run(['0'], fsm) == ['b'], "Unit test 1 failed"
    assert run(['0', '0'], fsm) == ['b', 'b'], "Unit test 2 failed"
    assert run(['0', '0', '0'], fsm) == ['b', 'b', 'a'], "Unit test 3 failed"
    assert run(['0', '0', '0', '0'], fsm) == ['b', 'b', 'a', 'a'], "Integration test 4 failed"
    assert run(['0', '0', '1', '1'], fsm) == ['b', 'b', 'a', 'b'], "Unit test 5 failed"

    print("All test cases passed")

test_fsm()


---